# who_mortality


In [ ]:
from __future__ import annotations

import re
from datetime import datetime, timezone

from pyspark.sql import functions as F, DataFrame
from pyspark.sql.types import (
    IntegerType, LongType, DoubleType, BooleanType, StringType, TimestampType,
)

SOURCE_SYSTEM   = "WHO Mortality Database"
COUNTRY_CODE    = "GTM"
COUNTRY_NAME    = "Guatemala"
REGION_CODE     = "CSA"
REGION_NAME     = "Central and South America"

LOW_COUNT_THRESHOLD   = 5
SUPPRESS_LOW_COUNTS   = False
MIN_REASONABLE_YEAR   = 1900
MAX_REASONABLE_YEAR   = datetime.now(timezone.utc).year + 1
MAX_REASONABLE_COUNT  = 1_000_000_000

# ── DIM_ETARIO grid canónico (docs/modelo_dimensional.md §5.2) ────────────────
# (id_etario, grupo_edad_codigo, grupo_edad_nombre, edad_min, edad_max, categoria_etaria)
_DIM_ETARIO_GRID = [
    (1,  "LT1",   "Menores de 1",      0, 0,    "Niñez"),
    (2,  "01-04", "1 a 4",             1, 4,    "Niñez"),
    (3,  "05-09", "5 a 9",             5, 9,    "Niñez"),
    (4,  "10-14", "10 a 14",          10, 14,   "Niñez"),
    (5,  "15-19", "15 a 19",          15, 19,   "Juventud"),
    (6,  "20-24", "20 a 24",          20, 24,   "Juventud"),
    (7,  "25-29", "25 a 29",          25, 29,   "Juventud"),
    (8,  "30-34", "30 a 34",          30, 34,   "Adultez"),
    (9,  "35-39", "35 a 39",          35, 39,   "Adultez"),
    (10, "40-44", "40 a 44",          40, 44,   "Adultez"),
    (11, "45-49", "45 a 49",          45, 49,   "Adultez"),
    (12, "50-54", "50 a 54",          50, 54,   "Adultez"),
    (13, "55-59", "55 a 59",          55, 59,   "Adultez"),
    (14, "60-64", "60 a 64",          60, 64,   "Adultez"),
    (15, "65-69", "65 a 69",          65, 69,   "Vejez"),
    (16, "70-74", "70 a 74",          70, 74,   "Vejez"),
    (17, "75-79", "75 a 79",          75, 79,   "Vejez"),
    (18, "80-84", "80 a 84",          80, 84,   "Vejez"),
    (19, "85+",   "85 y más",         85, None, "Vejez"),
    (98, "UNK",   "No especificada",  None, None, "No especificado"),
    (99, "ALL",   "Todas las edades", 0, None, "Total"),
]
# id_etario -> categoria_etaria (para roll-up Niñez/Juventud/Adultez/Vejez/Total)
_ETARIO_CAT_MAP = F.create_map(
    *[x for row in _DIM_ETARIO_GRID for x in (F.lit(row[0]), F.lit(row[5]))]
)
# WHO age_group_code -> id_etario (crosswalk §5.3). WHO deaths ya trae las 21
# bandas canónicas; population trae 28 (21 + 7 agregados solapados que se
# descartan en _standardize_population_frame, ver _WHO_AGE_DESCARTE).
_WHO_ETARIO_MAP = F.create_map(
    *[x for k, v in {
        "all": 99, "0": 1, "1-4": 2, "5-9": 3, "10-14": 4,
        "15-19": 5, "20-24": 6, "25-29": 7, "30-34": 8,
        "35-39": 9, "40-44": 10, "45-49": 11, "50-54": 12,
        "55-59": 13, "60-64": 14, "65-69": 15, "70-74": 16,
        "75-79": 17, "80-84": 18, "85+": 19, "age_unknown": 98,
    }.items() for x in (F.lit(k), F.lit(v))]
)
# Agregados solapados que coexisten con las hojas en population (§5.4 #1).
# Se descartan en staging para evitar doble conteo al sumar.
_WHO_AGE_DESCARTE = ("55-74", "75+", "age00_04", "age05_14",
                     "age15_24", "age25_34", "age35_54")

# ── Normalización de causa a GBD (docs/modelo_dimensional.md §6) ──────────────
# WHO Mortality usa su propia lista de CGXXXX (Nivel 1: Communicable /
# Noncommunicable / Injuries / Ill-defined), NO las 16 de Nivel 2 de IHME.
# Confirmado contra la data real: los CG de WHO no coinciden con los del
# diccionario GBD (p. ej. WHO CG0590 = "Noncommunicable diseases", pero en
# MAPEO_GBD CG0590 = "Neurological disorders"). Por eso gbd_code queda NULL
# (no se fuerza) y gbd_nombre se puebla con indicator_name (best-effort,
# igual que INE/Panamá para causas fuera de las 16, §6.4).
MAPEO_GBD = {
    "Neoplasms":                                    ("CG0600", 410),
    "Cardiovascular diseases":                      ("CG0530", 491),
    "Neglected tropical diseases and malaria":      ("CG0250", 344),
    "Nutritional deficiencies":                     ("CG0470", 386),
    "Diabetes and kidney diseases":                 ("CG0510", 955),
    "Mental disorders":                             ("CG0610", 545),
    "Neurological disorders":                       ("CG0590", 542),
    "Self-harm and interpersonal violence":         ("CG0860", 717),
    "COVID-19":                                     ("CG0995", 1013),
    "Transport injuries":                           ("CG0810", 688),
    "Digestive diseases":                           ("CG0580", 526),
    "Unintentional injuries":                       ("CG0850", 696),
    "Diarrheal diseases":                           ("CG0350", 302),
    "HIV/AIDS and sexually transmitted infections": ("CG0290", 366),
    "Respiratory infections and tuberculosis":      ("CG0390", 337),
    "Chronic respiratory diseases":                 ("CG0570", 508),
}
_WHO_CODES_16 = {v[0] for v in MAPEO_GBD.values()}  # set de CG de las 16 (validación)

WHO_AGE_GROUP_CODE_MAP = {
    "age_all":   "all",
    "age00":     "0",
    "age01_04":  "1-4",
    "age05_09":  "5-9",
    "age10_14":  "10-14",
    "age15_19":  "15-19",
    "age20_24":  "20-24",
    "age25_29":  "25-29",
    "age30_34":  "30-34",
    "age35_39":  "35-39",
    "age40_44":  "40-44",
    "age45_49":  "45-49",
    "age50_54":  "50-54",
    "age55_59":  "55-59",
    "age60_64":  "60-64",
    "age65_69":  "65-69",
    "age70_74":  "70-74",
    "age75_79":  "75-79",
    "age80_84":  "80-84",
    "age85_over":"85+",
    "age55_74":  "55-74",
    "age75_over":"75+",
}

WHO_AGE_GROUP_LABEL_MAP = {
    "all": "all", "[all]": "all",
    "0": "0", "[0]": "0",
    "1-4": "1-4", "[1-4]": "1-4",
    "5-9": "5-9", "[5-9]": "5-9",
    "10-14": "10-14", "[10-14]": "10-14",
    "15-19": "15-19", "[15-19]": "15-19",
    "20-24": "20-24", "[20-24]": "20-24",
    "25-29": "25-29", "[25-29]": "25-29",
    "30-34": "30-34", "[30-34]": "30-34",
    "35-39": "35-39", "[35-39]": "35-39",
    "40-44": "40-44", "[40-44]": "40-44",
    "45-49": "45-49", "[45-49]": "45-49",
    "50-54": "50-54", "[50-54]": "50-54",
    "55-59": "55-59", "[55-59]": "55-59",
    "60-64": "60-64", "[60-64]": "60-64",
    "65-69": "65-69", "[65-69]": "65-69",
    "70-74": "70-74", "[70-74]": "70-74",
    "75-79": "75-79", "[75-79]": "75-79",
    "80-84": "80-84", "[80-84]": "80-84",
    "85+": "85+", "[85+]": "85+",
    "55-74": "55-74", "[55-74]": "55-74",
    "75+": "75+", "[75+]": "75+",
}

SEX_MAP = {
    "both": "all", "bothsexes": "all",
    "male": "male", "males": "male",
    "female": "female", "females": "female",
    "all": "all",
    "unknown": "unknown",
}

INDICATOR_TABLES = [
    (
        "semi2.sandbox.raw_who_mortality_deaths_by_age_group_gtm",
        "stage.who_deaths_by_age_group_gtm",
    ),
]

POPULATION_TABLE = (
    "semi2.sandbox.raw_who_mortality_population_distribution_gtm",
    "stage.who_population_distribution_gtm",
)

INDICATOR_COLS_ORDERED = [
    "id",
    "source_system", "country_code", "country_name", "region_code", "region_name",
    "source_file", "source_url", "source_location",
    "source_export_date_raw", "source_export_ts",
    "table_kind", "privacy_class",
    "indicator_code", "indicator_name", "who_indicator_code", "icd10_group",
    "cie10_code", "cie10_nombre", "gbd_code", "gbd_nombre", "gbd_cause_id",
    "year",
    "sex", "es_masculino", "es_femenino", "es_total", "es_desconocido",
    "age_group_code", "age_group", "id_etario", "categoria_etaria",
    "number",
    "percentage_of_cause_specific_deaths_out_of_total_deaths",
    "age_standardized_death_rate_per_100_000_standard_population",
    "death_rate_per_100_000_population",
    "sensitive_cell_flag", "record_hash", "ingestion_ts",
]

POPULATION_COLS_ORDERED = [
    "id",
    "source_system", "country_code", "country_name", "region_code", "region_name",
    "source_file", "source_url", "source_location",
    "source_export_date_raw", "source_export_ts",
    "table_kind", "privacy_class",
    "year",
    "sex", "es_masculino", "es_femenino", "es_total", "es_desconocido",
    "age_group_code", "age_group", "id_etario", "categoria_etaria",
    "population",
    "sensitive_cell_flag", "record_hash", "ingestion_ts",
]

INDICATOR_SCHEMA = {
    "id":                     LongType(),
    "source_system":          StringType(),
    "country_code":           StringType(),
    "country_name":           StringType(),
    "region_code":            StringType(),
    "region_name":            StringType(),
    "source_file":            StringType(),
    "source_url":             StringType(),
    "source_location":        StringType(),
    "source_export_date_raw": StringType(),
    "source_export_ts":       TimestampType(),
    "table_kind":             StringType(),
    "privacy_class":          StringType(),
    "indicator_code":         StringType(),
    "indicator_name":         StringType(),
    "who_indicator_code":     StringType(),
    "icd10_group":            StringType(),
    "cie10_code":             StringType(),
    "cie10_nombre":           StringType(),
    "gbd_code":               StringType(),
    "gbd_nombre":             StringType(),
    "gbd_cause_id":           IntegerType(),
    "year":                   IntegerType(),
    "sex":                    StringType(),
    "es_masculino":           BooleanType(),
    "es_femenino":            BooleanType(),
    "es_total":               BooleanType(),
    "es_desconocido":         BooleanType(),
    "age_group_code":         StringType(),
    "age_group":              StringType(),
    "id_etario":              IntegerType(),
    "categoria_etaria":       StringType(),
    "number":                 DoubleType(),
    "sensitive_cell_flag":    BooleanType(),
    "record_hash":            StringType(),
    "ingestion_ts":           TimestampType(),
}

POPULATION_SCHEMA = {
    "id":                     LongType(),
    "source_system":          StringType(),
    "country_code":           StringType(),
    "country_name":           StringType(),
    "region_code":            StringType(),
    "region_name":            StringType(),
    "source_file":            StringType(),
    "source_url":             StringType(),
    "source_location":        StringType(),
    "source_export_date_raw": StringType(),
    "source_export_ts":       TimestampType(),
    "table_kind":             StringType(),
    "privacy_class":          StringType(),
    "year":                   IntegerType(),
    "sex":                    StringType(),
    "es_masculino":           BooleanType(),
    "es_femenino":            BooleanType(),
    "es_total":               BooleanType(),
    "es_desconocido":         BooleanType(),
    "age_group_code":         StringType(),
    "age_group":              StringType(),
    "id_etario":              IntegerType(),
    "categoria_etaria":       StringType(),
    "population":             DoubleType(),
    "sensitive_cell_flag":    BooleanType(),
    "record_hash":            StringType(),
    "ingestion_ts":           TimestampType(),
}


def _verify_schema(df: DataFrame, expected: dict, table_name: str,
                   strict: bool = True) -> None:
    actual = {f.name: f.dataType for f in df.schema.fields}
    errors = []
    for col, exp_type in expected.items():
        if col not in actual:
            if strict:
                errors.append(f"  MISSING  '{col}'")
        elif type(actual[col]) != type(exp_type):
            errors.append(f"  MISMATCH '{col}': expected {exp_type}, got {actual[col]}")
    extra = [c for c in actual if c not in expected]
    if extra:
        errors.append(f"  EXTRA    {extra}")
    if errors:
        print(f"[SCHEMA] {table_name} — {len(errors)} problema(s):")
        for msg in errors:
            print(msg)
    else:
        print(f"[SCHEMA] {table_name} — OK ({len(actual)} columnas, tipos correctos)")


def _map_values(df: DataFrame, column_name: str, mapping: dict[str, str]) -> DataFrame:
    expr = F.create_map(*[item for pair in mapping.items() for item in (F.lit(pair[0]), F.lit(pair[1]))])
    return df.withColumn(column_name, F.coalesce(F.element_at(expr, F.col(column_name)), F.col(column_name)))


def _normalize_column_names(df: DataFrame) -> DataFrame:
    # Normaliza los nombres de columnas de datos a snake_case. Las columnas de
    # metadata de ingestión (prefijo "_": _country_code, _source_file, etc.) se
    # dejan intactas porque _promote_raw_metadata las referencia por ese nombre.
    def _snake(value: str) -> str:
        text = re.sub(r"[^0-9A-Za-z]+", "_", value.strip())
        return re.sub(r"_+", "_", text).strip("_").lower()
    return df.toDF(*[c if c.startswith("_") else _snake(c) for c in df.columns])


def _normalize_string_columns(df: DataFrame) -> DataFrame:
    from pyspark.sql.types import StringType as ST
    result = df
    for field in df.schema.fields:
        if isinstance(field.dataType, ST):
            result = result.withColumn(
                field.name,
                F.when(
                    F.col(field.name).isNull() | (F.length(F.trim(F.col(field.name))) == 0),
                    F.lit(None).cast("string"),
                ).otherwise(F.regexp_replace(F.trim(F.col(field.name)), r"\s+", " ")),
            )
    return result


def _harmonize_sex(df: DataFrame) -> DataFrame:
    result = df.withColumn("sex", F.lower(F.trim(F.col("sex"))))
    result = _map_values(result, "sex", SEX_MAP)
    return result.withColumn("sex", F.when(F.col("sex").isNull(), F.lit("unknown")).otherwise(F.col("sex")))


def _harmonize_age_fields(df: DataFrame) -> DataFrame:
    result = df.withColumn("age_group_code", F.lower(F.trim(F.col("age_group_code"))))
    result = result.withColumn("age_group", F.lower(F.trim(F.col("age_group"))))
    result = _map_values(result, "age_group_code", WHO_AGE_GROUP_CODE_MAP)
    result = _map_values(result, "age_group", WHO_AGE_GROUP_LABEL_MAP)
    result = result.withColumn("age_group_code", F.when(F.col("age_group_code").isNull(), F.lit("unknown")).otherwise(F.col("age_group_code")))
    return result.withColumn("age_group", F.when(F.col("age_group").isNull(), F.col("age_group_code")).otherwise(F.col("age_group")))


def _sanitize_year(df: DataFrame) -> DataFrame:
    year = F.col("year").cast("int")
    return df.withColumn("year", F.when((year >= MIN_REASONABLE_YEAR) & (year <= MAX_REASONABLE_YEAR), year).otherwise(F.lit(None).cast("int")))


def _sanitize_non_negative(df: DataFrame, column_name: str, max_value: float = MAX_REASONABLE_COUNT) -> DataFrame:
    value = F.col(column_name).cast("double")
    return df.withColumn(column_name, F.when(value.isNull(), F.lit(None).cast("double")).when((value < 0) | (value > max_value), F.lit(None).cast("double")).otherwise(value))


def _sanitize_percentage(df: DataFrame, column_name: str) -> DataFrame:
    value = F.col(column_name).cast("double")
    return df.withColumn(column_name, F.when(value.isNull(), F.lit(None).cast("double")).when((value < 0) | (value > 100), F.lit(None).cast("double")).otherwise(value))


def _sanitize_rate(df: DataFrame, column_name: str) -> DataFrame:
    value = F.col(column_name).cast("double")
    return df.withColumn(column_name, F.when(value.isNull(), F.lit(None).cast("double")).when(value < 0, F.lit(None).cast("double")).otherwise(value))


def _add_sex_booleans(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("_sx", F.lower(F.trim(F.col("sex"))))
        .withColumn("es_masculino",
            F.when(F.col("_sx") == "male",   F.lit(True)).otherwise(F.lit(False)))
        .withColumn("es_femenino",
            F.when(F.col("_sx") == "female", F.lit(True)).otherwise(F.lit(False)))
        .withColumn("es_total",
            F.when(F.col("_sx") == "all",    F.lit(True)).otherwise(F.lit(False)))
        .withColumn("es_desconocido",
            F.when(F.col("_sx").isin("male", "female", "all"), F.lit(False))
             .otherwise(F.lit(True)))
        .drop("_sx")
    )


def _add_id(df: DataFrame) -> DataFrame:
    return df.withColumn("id", F.monotonically_increasing_id().cast(LongType()))


def _record_hash(df: DataFrame, key_columns: list[str]) -> DataFrame:
    values = [F.coalesce(F.col(column).cast("string"), F.lit("")) for column in key_columns]
    return df.withColumn("record_hash", F.sha2(F.concat_ws("||", *values), 256))


def _privacy_flags(df: DataFrame, count_column: str | None = None) -> DataFrame:
    result = df.withColumn("sensitive_cell_flag", F.lit(False))
    if count_column and count_column in result.columns:
        result = result.withColumn("sensitive_cell_flag",
            F.when(F.col(count_column).isNull(), F.lit(False))
             .otherwise(F.col(count_column) < LOW_COUNT_THRESHOLD))
        if SUPPRESS_LOW_COUNTS:
            result = result.withColumn(count_column,
                F.when(F.col(count_column) < LOW_COUNT_THRESHOLD, F.lit(None).cast("double"))
                 .otherwise(F.col(count_column)))
    return result


def _promote_raw_metadata(df: DataFrame) -> DataFrame:
    export_ts = (
        F.to_timestamp(F.col("_source_export_date"), "M/d/yyyy h:mm:ss a")
        if "_source_export_date" in df.columns
        else F.lit(None).cast("timestamp")
    )
    return (
        df
        .withColumn("source_system",          F.lit(SOURCE_SYSTEM))
        .withColumn("country_code",           F.coalesce(F.col("_country_code"),       F.lit(COUNTRY_CODE)))
        .withColumn("country_name",           F.coalesce(F.col("_country_name"),       F.lit(COUNTRY_NAME)))
        .withColumn("region_code",            F.coalesce(F.col("_region_code"),        F.lit(REGION_CODE)))
        .withColumn("region_name",            F.coalesce(F.col("_region_name"),        F.lit(REGION_NAME)))
        .withColumn("source_export_date_raw", F.col("_source_export_date"))
        .withColumn("source_export_ts",       export_ts)
        .withColumn("source_location",        F.col("_source_location"))
        .withColumn("source_file",            F.col("_source_file"))
        .withColumn("source_url",             F.col("_source_url"))
        .withColumn("table_kind",             F.col("_table_kind"))
        .withColumn("privacy_class",          F.lit("aggregated_public"))
        .withColumn("ingestion_ts",           F.col("_ingestion_timestamp"))
    )


def _standardize_indicator_frame(df: DataFrame) -> DataFrame:
    result = df.withColumn("indicator_code", F.upper(F.trim(F.col("indicator_code"))))
    result = result.withColumn("indicator_name", F.regexp_replace(F.trim(F.col("indicator_name")), r"\s+", " "))
    result = result.withColumn("sex", F.when(F.col("sex") == "", F.lit(None).cast("string")).otherwise(F.col("sex")))
    result = _harmonize_sex(result)
    result = result.withColumn("age_group_code", F.when(F.col("age_group_code") == "", F.lit(None).cast("string")).otherwise(F.col("age_group_code")))
    result = result.withColumn("age_group", F.when(F.col("age_group") == "", F.lit(None).cast("string")).otherwise(F.col("age_group")))
    result = _harmonize_age_fields(result)
    result = _sanitize_year(result)
    result = _sanitize_non_negative(result, "number")
    result = _sanitize_percentage(result, "percentage_of_cause_specific_deaths_out_of_total_deaths")
    result = _sanitize_rate(result, "age_standardized_death_rate_per_100_000_standard_population")
    result = _sanitize_rate(result, "death_rate_per_100_000_population")
    result = result.withColumn("who_indicator_code", F.col("indicator_code"))
    result = result.withColumn("icd10_group", F.lower(F.regexp_replace(F.col("indicator_name"), r"[^a-zA-Z0-9]+", "_")))
    result = result.withColumn("icd10_group", F.regexp_replace(F.col("icd10_group"), r"_+", "_"))
    result = result.withColumn("icd10_group", F.regexp_replace(F.col("icd10_group"), r"^_|_$", ""))
    result = result.withColumn("icd10_group", F.when(F.col("icd10_group") == "", F.lit(None).cast("string")).otherwise(F.col("icd10_group")))
    result = _privacy_flags(result, "number")
    result = _record_hash(result, ["indicator_code", "year", "sex", "age_group_code", "age_group"])
    return result.dropDuplicates(["indicator_code", "year", "sex", "age_group_code", "age_group"])


def _standardize_population_frame(df: DataFrame) -> DataFrame:
    result = df.withColumn("age_group_code", F.when(F.col("age_group_code") == "", F.lit(None).cast("string")).otherwise(F.col("age_group_code")))
    result = result.withColumn("age_group", F.when(F.col("age_group") == "", F.lit(None).cast("string")).otherwise(F.col("age_group")))
    result = result.withColumn("sex", F.when(F.col("sex") == "", F.lit(None).cast("string")).otherwise(F.col("sex")))
    result = _harmonize_age_fields(result)
    result = _harmonize_sex(result)
    result = _sanitize_year(result)
    result = _sanitize_non_negative(result, "population")
    result = _privacy_flags(result)
    result = _record_hash(result, ["year", "sex", "age_group_code", "age_group"])
    return result.dropDuplicates(["year", "sex", "age_group_code", "age_group"])


def _add_gbd_columns(df: DataFrame) -> DataFrame:
    # WHO Mortality trae indicator_code (CGXXXX) que es la lista de causas
    # propia de WHO (Nivel 1: Communicable/Noncommunicable/Injuries/Ill-defined),
    # NO las 16 de Nivel 2 de IHME. Los CG de WHO no coinciden con los del
    # diccionario GBD (p. ej. WHO CG0590 = "Noncommunicable diseases" pero en
    # MAPEO_GBD CG0590 = "Neurological disorders"). Por eso gbd_code queda NULL
    # (no se fuerza a las 16) y gbd_nombre se puebla con indicator_name
    # (best-effort, igual que INE/Panamá para fuera-de-16, §6.4). cie10 no aplica
    # (WHO no trae CIE-10 fino; icd10_group es un slug derivado de indicator_name).
    return (
        df
        .withColumn("cie10_code",   F.lit(None).cast(StringType()))
        .withColumn("cie10_nombre", F.lit(None).cast(StringType()))
        .withColumn("gbd_code",     F.lit(None).cast(StringType()))
        .withColumn("gbd_nombre",   F.col("indicator_name"))
        .withColumn("gbd_cause_id", F.lit(None).cast(IntegerType()))
    )


def _add_etario_columns(df: DataFrame) -> DataFrame:
    # WHO age_group_code -> id_etario (crosswalk §5.3). Los age_group_code ya
    # están normalizados a las etiquetas canónicas (all, 0, 1-4, ... 85+,
    # age_unknown) por _harmonize_age_fields. age_unknown -> 98 (UNK), all -> 99.
    return (
        df
        .withColumn("id_etario",
            _WHO_ETARIO_MAP[F.col("age_group_code")].cast(IntegerType()))
        .withColumn("categoria_etaria", _ETARIO_CAT_MAP[F.col("id_etario")])
    )


def _descartar_agregados_solapados(df: DataFrame) -> DataFrame:
    # Population trae bandas hoja (0, 1-4, 5-9, ... 85+) + agregados que las
    # solapan (age00_04 = 0+1-4, 55-74 = 55-59+60-64+65-69+70-74, etc.). Si se
    # conservan, sumar duplica. Se descartan en staging (§5.4 #1).
    return df.filter(~F.col("age_group_code").isin(*_WHO_AGE_DESCARTE))


def transform_indicator(df: DataFrame) -> DataFrame:
    df = _normalize_column_names(df)
    df = _normalize_string_columns(df)
    df = _standardize_indicator_frame(df)
    df = _add_gbd_columns(df)
    df = _add_etario_columns(df)
    df = _promote_raw_metadata(df)
    df = _add_sex_booleans(df)
    df = _add_id(df)
    return df.select(*[c for c in INDICATOR_COLS_ORDERED if c in df.columns])


def transform_population(df: DataFrame) -> DataFrame:
    df = _normalize_column_names(df)
    df = _normalize_string_columns(df)
    df = _standardize_population_frame(df)
    df = _descartar_agregados_solapados(df)
    df = _add_etario_columns(df)
    df = _promote_raw_metadata(df)
    df = _add_sex_booleans(df)
    df = _add_id(df)
    return df.select(*[c for c in POPULATION_COLS_ORDERED if c in df.columns])


spark.sql("CREATE SCHEMA IF NOT EXISTS stage")

try:
    for src, dst in INDICATOR_TABLES:
        raw    = spark.read.table(src)
        staged = transform_indicator(raw)
        _verify_schema(staged, INDICATOR_SCHEMA, dst, strict=False)
        display(staged)
        staged.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(dst)

    src_pop, dst_pop = POPULATION_TABLE
    raw_pop    = spark.read.table(src_pop)
    staged_pop = transform_population(raw_pop)
    _verify_schema(staged_pop, POPULATION_SCHEMA, dst_pop)
    display(staged_pop)
    staged_pop.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(dst_pop)

except Exception as e:
    print(f"Error al procesar sandbox.raw_who_mortality_*: {e}")
    raise
